<center><b>CTS2000: Scripting for the Humanities</b></center>
<center>8-2 : Preparing the Newswire data</center>

---

This notebook shows to download the **Newswire** dataset from Hugging Face and turn it into the two small files we use in **8-3: Data Visualization**. You don't need to run it: it's here so you can see exactly where our data comes from, what is kept and what is left out, and so you can adapt it if you want to use Hugging Face data later.

Run this on **Google Colab**, not on your laptops: the full dataset is over 5 GB, and Colab gives us a free computer in the cloud with plenty of space and a fast connection. It also already has the libraries we need installed and our class environment does not.

📄 Dell et al. (2024), [*Newswire: A Large-Scale Structured Database of a Century of Historical News*](https://arxiv.org/abs/2406.09490). Data on [Hugging Face](https://huggingface.co/datasets/dell-research-harvard/newswire), CC-BY licence.

<a target="_blank" href="https://colab.research.google.com/github/mervetekgurler/cts2000_fall2026/blob/main/units/week_8/8-2_newswire_data_prep.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

## 1. Hugging Face and the dataset card

[**Hugging Face**](https://huggingface.co/) is a platform where researchers share **models** and **datasets**. Every dataset has a **dataset card**: a page that explains what's in the data, how it was made, its licence and its limits. Before using any dataset, read the card. It's the first step of the dataset evaluation we do for final projects.

On the Newswire card, under **Files and versions**, we can see that the dataset is stored as **one JSON file per year**, from `1878_data_clean.json` to `1977_data_clean.json`. That means we can download one year at a time.

## 2. Setup

Colab already has the Hugging Face `datasets` library installed. This cell just checks the version:

In [ ]:
import os
import gzip
import json

import pandas as pd
import datasets
from datasets import load_dataset
from huggingface_hub import list_repo_files

print("datasets", datasets.__version__)

In [ ]:
# ---- settings ----
REPO = "dell-research-harvard/newswire"
YEARS = None                # None = every year; or a list, like [1930, 1931, 1932]
SAMPLE_PER_YEAR = 2000      # None = keep every story
SAVE_TO = "newswire/"       # a folder on the Colab computer

# the fields we keep; everything else is dropped
KEEP = ["year", "dates", "article", "ca_topic",
        "wire_city", "wire_state", "wire_country", "wire_coordinates", "wire_location_notes",
        "people_mentioned", "newspaper_metadata"]

`SAMPLE_PER_YEAR` -> the full dataset has 2.7 million stories, which is too much for our laptops in class. So we keep a **random sample** of up to 2,000 stories per year. That's a choice, and we'll come back to what it means.

## 3. Looking at one year

Let's start with one year, to see what a record looks like. `load_dataset` downloads the file from Hugging Face and loads it:

In [ ]:
one_year = load_dataset(REPO, data_files=["1880_data_clean.json"], split="train")
one_year

`load_dataset(REPO, data_files=[...])` -> downloads only the files we ask for, instead of the whole 5 GB.

`split="train"` -> many datasets are divided into a training set and a test set, for machine learning. Newswire isn't, so all the stories are in one split called `"train"`.

The result is a Hugging Face `Dataset`. It works a bit like a list of dictionaries, one per story:

In [ ]:
first_story = one_year[0]
first_story.keys()

In [ ]:
first_story["article"][:500]

In [ ]:
print(first_story["ner_words"][:15])
print(first_story["ner_labels"][:15])

Look at `ner_words` and `ner_labels`: the whole article, **again**, one token at a time, with a NER label for every token. Most labels are `O`, for tokens that aren't part of any entity. It's the same **BIO** idea as Stanza's `B-PERSON` and `E-PERSON`: `B-` begins an entity, `I-` continues it.

Storing every token twice, plus a label for each one, is a big part of why the dataset is so large. We don't need it for our charts, especially since the people have already been extracted into `people_mentioned`:

In [ ]:
first_story["people_mentioned"]

In [ ]:
first_story["newspaper_metadata"][:3]

## 4. What we keep

| Field | What it is |
|---|---|
| `year`, `dates` | when the story was published (`dates` is a list: one date for each day it ran) |
| `article` | the text of the story |
| `ca_topic` | the story's topic, predicted by a classifier trained on the *Comparative Agendas* topics |
| `wire_city`, `wire_state`, `wire_country`, `wire_coordinates` | where the story was written |
| `wire_location_notes` | for stories not written in a city, like *War Front (WWII)* |
| `people_mentioned` | the people in the story, linked to Wikidata, with gender and occupation |
| `newspaper_metadata` | every newspaper that printed the story, with its city and state |

We drop the NER tokens and labels, the byline, the seven yes/no topic tags, and `cluster_size`, which is just the number of newspapers in `newspaper_metadata`.

## 5. Download, slim down and sample every year

Now we go through the year files one at a time. For each year, we:

1. download it with `load_dataset`,
2. keep only our columns with `select_columns`,
3. shuffle it and keep the first `SAMPLE_PER_YEAR` stories, which gives us a random sample,
4. write the stories to one compressed **JSON lines** file, and
5. note how many stories the year had **before** sampling.

In [ ]:
year_files = sorted(f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith("_data_clean.json"))
if YEARS:
    year_files = [f for f in year_files if int(f.split("_")[0]) in YEARS]
print(len(year_files), "year files, from", year_files[0], "to", year_files[-1])

In [ ]:
os.makedirs(SAVE_TO, exist_ok=True)
counts = []

with gzip.open(SAVE_TO + "newswire_sample.jsonl.gz", "wt", encoding="utf-8") as out:
    for filename in year_files:
        year = int(filename.split("_")[0])
        stories = load_dataset(REPO, data_files=[filename], split="train")
        n_stories = len(stories)

        # keep only our columns (if a year is missing one, we skip it rather than crash)
        stories = stories.select_columns([c for c in KEEP if c in stories.column_names])

        # a random sample: shuffle, then take the first SAMPLE_PER_YEAR stories
        if SAMPLE_PER_YEAR and n_stories > SAMPLE_PER_YEAR:
            stories = stories.shuffle(seed=42).select(range(SAMPLE_PER_YEAR))

        for story in stories:
            out.write(json.dumps(story) + "\n")

        counts.append({"year": year, "stories_in_dataset": n_stories, "stories_kept": len(stories)})
        print(f"{year}: {n_stories:,} stories, kept {len(stories):,}")

counts = pd.DataFrame(counts)
counts.to_csv(SAVE_TO + "newswire_counts_by_year.csv", index=False)

print(f"\n{counts['stories_kept'].sum():,} stories kept, out of {counts['stories_in_dataset'].sum():,}")
print(f"file size: {os.path.getsize(SAVE_TO + 'newswire_sample.jsonl.gz') / 1e6:,.0f} MB")

Let's unpack this:

`stories.select_columns([...])` -> keeps only the columns we listed.

`stories.shuffle(seed=42).select(range(SAMPLE_PER_YEAR))` -> `shuffle` puts the stories in a random order, and `select(range(2000))` keeps the first 2,000 of them, so together they give us a random sample. `seed=42` makes the "random" order the same every time we run this, so anyone can recreate exactly our sample.

`json.dumps(story)` -> turns one story, a dictionary, into a line of JSON text. Writing one story per line is called **JSON lines**.

`gzip.open(..., "wt")` -> opens a **compressed** file for writing text. Our file ends in `.jsonl.gz`, and pandas can read it directly with `pd.read_json(..., lines=True)`.

Why JSON and not CSV? `people_mentioned` and `newspaper_metadata` are lists of dictionaries. A CSV would turn them into plain strings, like the Montagu entities, and we'd need `ast.literal_eval` to turn them back.

Why save the counts separately? Because we take the same number of stories from almost every year, our sample can't show how much the dataset grows over time. `newswire_counts_by_year.csv` keeps the **real** numbers.

<div style="background: rgba(245, 180, 80, 0.15); border-left: 4px solid rgba(214, 140, 40, 0.9); border-radius: 6px; padding: 0.8em 1.2em; margin: 1.5em 0;">

⚠️ **A sample is a choice.** A random sample should look like the whole dataset for each year, but rare topics and rarely mentioned people may disappear from it. And since early years have fewer than 2,000 stories, we keep **all** of them, while later years are sampled. Keep this in mind when you compare years.

</div>

## 6. Save the files

Colab's computer is wiped when the session ends, so we need to download our two files:

In [ ]:
from google.colab import files

files.download(SAVE_TO + "newswire_sample.jsonl.gz")
files.download(SAVE_TO + "newswire_counts_by_year.csv")

We put both files in `data/newswire/` in our course folder, which is where **8-2: Data Visualization** loads them from.

💡 For bigger files, it's often easier to save to Google Drive instead: `from google.colab import drive`, then `drive.mount("/content/drive")`, and copy the files into a folder in your Drive.